# BÀI TẬP 1: TÍNH VÀ PHÂN TÍCH BIỂU ĐỒ ACF & PACF
**Môn học:** Quản lý & Khai phá Chuỗi Thời Gian (QLCTG)  
**Nhóm thực hiện:** Nhóm 07  
**Dữ liệu thực nghiệm:** `data/raw/AirPassengers.csv` (1949 - 1960)

---
### Mục tiêu bài tập:
1. Tính toán hệ số tự tương quan (ACF) và tự tương quan riêng phần (PACF) đến 40 lags.
2. Vẽ biểu đồ ACF/PACF kèm dải tin cậy 95% Bartlett $\pm rac{1.96}{\sqrt{n}}$.
3. Nhận diện tính chất tự tương quan, xu hướng (Trend) và tính mùa vụ chu kỳ 12 tháng (Seasonality).


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Cấu hình đồ họa
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# Đường dẫn dữ liệu
DATA_PATH = Path("../data/raw/AirPassengers.csv")
FIGURES_DIR = Path("../outputs/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


### Bước 1: Nạp và kiểm định chuỗi dữ liệu


In [ ]:
df = pd.read_csv(DATA_PATH)
df['Month'] = pd.to_datetime(df['Month'])
df.set_index('Month', inplace=True)
df.sort_index(inplace=True)

series = df['Passengers'].astype(float)
print(f"Tổng số quan sát: {len(series)}")
print(f"Khoảng thời gian: {series.index.min().strftime('%Y-%m')} -> {series.index.max().strftime('%Y-%m')}")
series.head()


### Bước 2: Tính toán ACF và PACF trên 40 lags


In [ ]:
nlags = 40
acf_vals = acf(series, nlags=nlags, alpha=0.05)
pacf_vals = pacf(series, nlags=nlags, alpha=0.05, method='yw')

n = len(series)
bartlett_ci = 1.96 / np.sqrt(n)

print(f"Dải tin cậy Bartlett 95%: +/- {bartlett_ci:.4f}")
print(f"ACF tại Lag 1: {acf_vals[0][1]:.4f}")
print(f"ACF tại Lag 12 (Đỉnh mùa vụ 1 năm): {acf_vals[0][12]:.4f}")
print(f"PACF tại Lag 1: {pacf_vals[0][1]:.4f}")


### Bước 3: Trực quan hóa ACF và PACF


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

plot_acf(series, lags=nlags, ax=axes[0], alpha=0.05, title="Autocorrelation Function (ACF) - AirPassengers (40 Lags)")
axes[0].set_xlabel("Độ trễ (Lags - Tháng)")
axes[0].set_ylabel("Hệ số ACF")
axes[0].grid(True, linestyle="--", alpha=0.6)

plot_pacf(series, lags=nlags, ax=axes[1], alpha=0.05, method="yw", title="Partial Autocorrelation Function (PACF) - AirPassengers (40 Lags)")
axes[1].set_xlabel("Độ trễ (Lags - Tháng)")
axes[1].set_ylabel("Hệ số PACF")
axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
fig_output = FIGURES_DIR / "bai1_acf_pacf.png"
plt.savefig(fig_output, dpi=300)
plt.show()
print(f"Đã lưu biểu đồ tại: {fig_output}")


### Bước 4: Nhận xét và Kết luận
1. **Tự tương quan rất mạnh:** ACF tại Lag 1 đạt 0.9480 và suy giảm rất từ từ, thể hiện chuỗi có quán tính lớn và chưa dừng (Non-stationary). Cần lấy sai phân bậc 1 $d=1$.
2. **Tính mùa vụ chu kỳ 12 tháng:** Xuất hiện các đỉnh nhô cao vượt dải tin cậy lặp lại đều đặn tại các bội số của 12 (Lag 12, 24, 36).
3. **Định hướng mô hình:** PACF cắt cụt ngay sau Lag 1 và có xung tại Lag 12/13, gợi ý mô hình SARIMA với $p=1, P=1$.
